# Notebook 3: GEE destruction-score map and building diagnostics

This notebook recreates the frozen final pipeline selected in Notebook 2 and
produces two complementary outputs:

1. a Google Earth Engine satellite map styled after the supplied reference —
   dense prediction footprints, green low scores, red high scores, and colors
   binned by within-map decile — plus a clean, high-resolution PNG; and
2. a Lonboard building map for granular error analysis with hover/click
   attributes and explicit TP, TN, FP, and FN categories.

The visualization does not refit a model, threshold, or score. Deciles affect
color only; `prob` remains the frozen model output. The local raster map and
before/after slider from the previous implementation are intentionally removed.


## 1. Setup and experiment selection


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q earthengine-api folium lonboard pyarrow geopandas rasterio xgboost


In [ ]:
import base64
import importlib.util
import io
import json
import sys
from pathlib import Path

import ee
import folium
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import torch
import torch.nn as nn
import torch.nn.functional as functional
import torchvision
import tensorflow as tf
from folium.plugins import Fullscreen
from lonboard import Map as LonboardMap, PolygonLayer
from PIL import Image
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from torch.utils.data import DataLoader

PROJECT_ROOT = Path('/content/drive/MyDrive/War-Damage-Detection')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Select one completed Notebook 2 experiment. Sensor and model family are
# inferred from its frozen final_selection.json.
EXPERIMENT_NAME = 'planet_exp001_base_cnn_spatial_roc_auc_compact_xgb'
SENSOR = 'auto'       # 'auto', 'sentinel_sar', or 'planet'
CITY = 'Gaza'         # used by Sentinel experiments
DATE = None           # None selects the latest registered Sentinel date

# Earth Engine and figure settings.
GEE_PROJECT = 'test-cnn-war-damage'
GEE_LOOKBACK_DAYS = 90
MAP_PAD_FRACTION = 0.06
FIGURE_WIDTH_PX = 2200
OVERLAY_OPACITY = 0.78
PANEL_LABEL = 'C'     # set to None for a standalone map

# Low-score green through high-score red, in within-map deciles.
REFERENCE_PALETTE = [
    '#007A4D', '#16945D', '#46AA70', '#7DBB82', '#B3CB94',
    '#D8C59D', '#E9A19A', '#EC8188', '#E86776', '#D94D61',
]

SENSOR_DIRS = {
    'sentinel_sar': PROJECT_ROOT / 'sentinel_sar',
    'planet': PROJECT_ROOT / 'planet',
}

if SENSOR == 'auto':
    matches = [name for name, folder in SENSOR_DIRS.items()
               if (folder / 'experiments' / EXPERIMENT_NAME).is_dir()]
    if len(matches) != 1:
        raise RuntimeError(
            f'Expected one experiment named {EXPERIMENT_NAME!r}, found {matches}. '
            'Set SENSOR explicitly if the name exists under both sensors.')
    SENSOR = matches[0]
elif SENSOR not in SENSOR_DIRS:
    raise ValueError(f'SENSOR must be one of auto, {sorted(SENSOR_DIRS)}')

SENSOR_DIR = SENSOR_DIRS[SENSOR]
EXPERIMENT_ROOT = SENSOR_DIR / 'experiments' / EXPERIMENT_NAME
if not EXPERIMENT_ROOT.is_dir():
    available = sorted(
        p.name for p in (SENSOR_DIR / 'experiments').glob('*') if p.is_dir())
    raise FileNotFoundError(
        f'Experiment not found: {EXPERIMENT_ROOT}. Available: {available}')


def load_json(path, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f'Missing experiment artifact: {path}')
        return None
    with path.open(encoding='utf-8') as handle:
        return json.load(handle)


def resolve_artifact(value, default_folder='models'):
    """Resolve current relative/absolute artifacts and legacy absolute paths."""
    if value in (None, '', 'None'):
        return None
    supplied = Path(value)
    candidates = [supplied]
    if not supplied.is_absolute():
        candidates.extend([
            EXPERIMENT_ROOT / supplied,
            EXPERIMENT_ROOT / default_folder / supplied,
        ])
    candidates.append(EXPERIMENT_ROOT / default_folder / supplied.name)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f'Could not resolve artifact {value!r}. Checked: '
        + ', '.join(map(str, candidates)))


def load_pipeline(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


pipeline = load_pipeline(f'{SENSOR}_map_pipeline', SENSOR_DIR / 'pipeline.py')
CONFIG = load_json(EXPERIMENT_ROOT / 'config.json')
FINAL_SELECTION = load_json(EXPERIMENT_ROOT / 'metrics' / 'final_selection.json')
FINAL_THRESHOLD = float(FINAL_SELECTION['threshold'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if SENSOR == 'sentinel_sar':
    FAMILY = 'scratch'
elif 'run' in FINAL_SELECTION and 'variant' in FINAL_SELECTION:
    FAMILY = 'scratch'
elif str(FINAL_SELECTION.get('saved_model', '')).endswith('.pt'):
    FAMILY = 'rsp_resnet50'
elif str(FINAL_SELECTION.get('saved_model', '')).endswith('.keras'):
    FAMILY = 'efficientnet_b0'
else:
    raise RuntimeError(
        f'Unrecognized Planet selection schema: {sorted(FINAL_SELECTION)}')

print(f'sensor: {SENSOR}')
print(f'experiment: {EXPERIMENT_NAME}')
print(f'model family: {FAMILY}')
print(f'validation-fitted threshold: {FINAL_THRESHOLD:.4f}')


## 2. Recreate the frozen pipeline and score all footprints

This is deployment scoring over the complete selected city/date. The resulting
GeoJSON is also the common input to the GEE and Lonboard views below.


In [ ]:
def score_sentinel_experiment(city, date):
    selection = FINAL_SELECTION
    tag = selection['run']
    variant = selection['variant']
    checkpoint_path = resolve_artifact(f'{tag}.pt')
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False)
    if not checkpoint.get('complete'):
        raise RuntimeError(f'Incomplete checkpoint: {checkpoint_path}')
    cfg = checkpoint['config']
    model = pipeline.build_model(
        checkpoint['model'], checkpoint['channel_names'], device=device,
        quiet=True, **checkpoint['params'])
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()

    stacker = None
    if variant.startswith('xgb_'):
        saved = torch.load(
            resolve_artifact('stackers.pt'), map_location='cpu',
            weights_only=False)
        if saved.get('format') not in (1, 2) or saved.get('config') != cfg:
            raise RuntimeError('stackers.pt does not match the selected CNN')
        stacker = saved['stackers'][tag][variant]

    if city not in pipeline.CITY_REGISTRY:
        raise KeyError(f'Unknown city {city!r}')
    date = date or pipeline.CITY_REGISTRY[city]['label_dates'][-1]
    if date not in pipeline.CITY_REGISTRY[city]['label_dates']:
        raise ValueError(f'{date} is not a registered assessment for {city}')
    data = pipeline.load_city(city)

    offsets = pipeline.temporal_offsets() if (
        stacker and stacker['temporal']) else [0]
    scores = {}
    for offset in offsets:
        score_date = date if offset == 0 else pipeline.shift_date(date, offset)
        source, rows = pipeline.city_source(
            data, score_date, cfg['features'], window_days='auto')
        full = np.full(len(data['table']), np.nan, np.float32)
        full[rows] = pipeline.predict_probs(
            model, source, mu=checkpoint['mu'], sd=checkpoint['sd'],
            device=device, batch=cfg.get('predict_batch', 512))
        scores[offset] = full

    base = scores[0]
    usable = np.isfinite(base)
    if variant == 'cnn':
        final_scores = base
    elif variant == 'cnn+fixed_neighbor_rule':
        final_scores = np.full_like(base, np.nan)
        smooth = cfg['spatial_smoothing']
        final_scores[usable] = pipeline.spatial_smooth(
            data['xy'][usable], base[usable],
            k=smooth['k'], weight=smooth['weight'])
    else:
        spatial = pipeline.neighbour_features(
            data['xy'], np.nan_to_num(base, nan=0.0),
            ks=tuple(cfg['stacking']['ks']))
        features = spatial
        if stacker['temporal']:
            features = pd.concat(
                [spatial, pipeline.temporal_features(scores)], axis=1)
        final_scores = np.full_like(base, np.nan)
        final_scores[usable] = stacker['model'].predict_proba(
            features.loc[usable, stacker['columns']])[:, 1]

    rows = np.flatnonzero(usable)
    table = data['table'].iloc[rows].copy()
    labels = pipeline.label_matrix(data['table'], data['dates'])
    if cfg.get('label_temporal', False):
        labels = pipeline.propagate_labels(labels)
    date_index = data['dates'].index(date)
    result = table[['geometry']].copy()
    result['prob'] = final_scores[rows].astype(float)
    result['cnn_prob'] = base[rows].astype(float)
    result['pred'] = (result['prob'] >= FINAL_THRESHOLD).astype(int)
    result['class'] = labels[rows, date_index].astype(int)
    result['split'] = pipeline.split_assignment(
        data['lat'], city, pipeline.strip_dates(cfg['split']))[rows]
    result['city'], result['date'] = city, date
    result['pipeline'] = f'{EXPERIMENT_NAME}: {tag} / {variant}'
    return result, date


def score_planet_experiment():
    normalization = load_json(
        EXPERIMENT_ROOT / 'metrics' / 'normalization.json')
    bands = list(normalization['bands'])
    lo = np.asarray(normalization['lo'], np.float32)
    hi = np.asarray(normalization['hi'], np.float32)
    data = pipeline.load_dataset(load_images=True)
    images, labels = data['X'], data['y']
    table = data['gdf'].reset_index(drop=True)
    if not np.array_equal(
            data['system_index'], table['system:index'].astype(str).to_numpy()):
        raise RuntimeError('Planet NPZ and parquet rows are not aligned')

    all_rows = np.arange(len(labels))
    loader = DataLoader(
        pipeline.PlanetPairDataset(
            images, labels, all_rows, bands=bands, lo=lo, hi=hi),
        batch_size=int(CONFIG.get('training', {}).get(
            'predict_batch_size', CONFIG.get('batch_size', 512))),
        shuffle=False, num_workers=0)
    cnn_scores = None

    if FAMILY == 'scratch':
        checkpoint_path = resolve_artifact(FINAL_SELECTION['checkpoint'])
        checkpoint = torch.load(
            checkpoint_path, map_location=device, weights_only=False)
        model = pipeline.build_planet_model(
            checkpoint['model_name'], n_channels=2 * len(bands),
            **checkpoint['model_params']).to(device)
        model.load_state_dict(checkpoint['model_state'])
        model.eval()
        cnn_scores, _ = pipeline.predict_planet_probs(model, loader, device)
        if FINAL_SELECTION['variant'] == 'cnn':
            final_scores = cnn_scores
        else:
            stacker = joblib.load(resolve_artifact(FINAL_SELECTION['stacker']))
            xy = table[['centroid_x', 'centroid_y']].to_numpy(np.float64)
            features = pipeline.neighbour_features(
                xy, cnn_scores, ks=CONFIG['stacking']['ks'])
            final_scores = stacker.predict_proba(features)[:, 1]

    elif FAMILY == 'rsp_resnet50':
        class RspSiameseDamage(nn.Module):
            def __init__(self, backbone, resize_to, selected_bands):
                super().__init__()
                if list(selected_bands) != ['R', 'G', 'B']:
                    raise ValueError('RSP ResNet-50 expects bands in RGB order')
                self.backbone = backbone
                self.resize_to = int(resize_to)
                self.half_channels = len(selected_bands)
                self.register_buffer(
                    'mean', torch.tensor((.485, .456, .406)).view(1, 3, 1, 1))
                self.register_buffer(
                    'std', torch.tensor((.229, .224, .225)).view(1, 3, 1, 1))
                self.head = nn.Sequential(
                    nn.Dropout(.4), nn.Linear(3 * 2048, 256),
                    nn.ReLU(inplace=True), nn.Dropout(.3), nn.Linear(256, 1))

            def encode(self, image):
                image = functional.interpolate(
                    image, size=(self.resize_to, self.resize_to),
                    mode='bilinear', align_corners=False)
                return self.backbone((image - self.mean) / self.std)

            def forward(self, images):
                pre = images[:, :self.half_channels]
                post = images[:, self.half_channels:]
                features = self.encode(torch.cat([pre, post], dim=0))
                before, after = features.chunk(2, dim=0)
                joined = torch.cat(
                    [before, after, torch.abs(after - before)], dim=1)
                return self.head(joined).squeeze(1)

        saved = torch.load(
            resolve_artifact(FINAL_SELECTION['saved_model']),
            map_location=device, weights_only=False)
        backbone = torchvision.models.resnet50(weights=None)
        backbone.fc = nn.Identity()
        model = RspSiameseDamage(
            backbone, CONFIG['resize_to'], bands).to(device)
        model.load_state_dict(saved['model_state'])
        model.eval()
        final_scores, _ = pipeline.predict_planet_probs(model, loader, device)

    elif FAMILY == 'efficientnet_b0':
        keras_model = tf.keras.models.load_model(
            resolve_artifact(FINAL_SELECTION['saved_model']))
        pre_channels, post_channels = pipeline.planet_band_indices(bands)
        scale = np.maximum(hi - lo, 1e-6)
        chunks = []
        for start in range(0, len(labels), 512):
            patch = np.asarray(images[start:start + 512], np.float32)
            pre = np.transpose(patch[:, list(pre_channels)], (0, 2, 3, 1))
            post = np.transpose(patch[:, list(post_channels)], (0, 2, 3, 1))
            pre = np.clip((pre - lo) / scale, 0, 1)
            post = np.clip((post - lo) / scale, 0, 1)
            chunks.append(keras_model.predict([pre, post], verbose=0).ravel())
        final_scores = np.concatenate(chunks)
    else:
        raise RuntimeError(FAMILY)

    split = pipeline.latitude_quantile_spatial_split(
        table,
        bands={
            role: tuple(CONFIG['split'][role])
            for role in ('train', 'stack', 'val', 'test')
        },
        reference=pipeline.footprint_latitude_reference())
    roles = np.full(len(table), 'unused', dtype='U10')
    for role in ('train', 'stack', 'val', 'test'):
        roles[split[role]] = 'val' if (
            FAMILY != 'scratch' and role == 'stack') else role

    result = table[['geometry']].copy()
    result['prob'] = np.asarray(final_scores, float)
    if cnn_scores is not None:
        result['cnn_prob'] = np.asarray(cnn_scores, float)
    result['pred'] = (result['prob'] >= FINAL_THRESHOLD).astype(int)
    result['class'] = labels.astype(int)
    result['split'] = roles
    result['city'], result['date'] = pipeline.CITY, pipeline.MAY_DATE
    result['pipeline'] = f'{EXPERIMENT_NAME} ({FAMILY})'
    return result, pipeline.MAY_DATE


if SENSOR == 'sentinel_sar':
    predictions, MAP_DATE = score_sentinel_experiment(CITY, DATE)
else:
    predictions, MAP_DATE = score_planet_experiment()
    CITY = str(predictions['city'].iloc[0])

predictions['threshold'] = FINAL_THRESHOLD
prediction_path = (
    EXPERIMENT_ROOT / 'predictions' /
    f'{CITY}_{MAP_DATE}_{EXPERIMENT_NAME}_map.geojson')
prediction_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_file(prediction_path, driver='GeoJSON')
print(f'wrote {len(predictions):,} buildings to {prediction_path}')
predictions.head(3)


## 3. GEE overview in reference style

Sentinel-2 surface reflectance supplies the dated satellite background through
Earth Engine. Every prediction footprint is painted by its **within-map score
decile**: deep green is the lowest decile and deep red is the highest. This
matches the visual encoding in the reference more closely than a binary
threshold map. The threshold remains available in the diagnostic map.


In [ ]:
def initialize_earth_engine():
    """Authenticate once, then initialize the configured Cloud project."""
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)


def padded_bounds(frame, fraction=0.06):
    """Return a compact WGS84 map extent with space around the predictions."""
    west, south, east, north = map(float, frame.to_crs(4326).total_bounds)
    dx = max((east - west) * float(fraction), 0.002)
    dy = max((north - south) * float(fraction), 0.002)
    return west - dx, south - dy, east + dx, north + dy


def sentinel2_composite(bounds, target_date, lookback_days=90):
    """Create a cloud-masked, muted true-colour GEE background."""
    initialize_earth_engine()
    target = pd.to_datetime(str(target_date), format='%Y%m%d')
    start = target - pd.Timedelta(days=int(lookback_days))
    end = target + pd.Timedelta(days=1)
    west, south, east, north = bounds
    aoi = ee.Geometry.Rectangle([west, south, east, north], geodesic=False)

    def mask_clouds(image):
        scl = image.select('SCL')
        clear = (scl.gt(1).And(scl.neq(3)).And(scl.neq(8))
                 .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
        return image.updateMask(clear).select(['B4', 'B3', 'B2'])

    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(aoi)
                  .filterDate(start.strftime('%Y-%m-%d'),
                              end.strftime('%Y-%m-%d'))
                  .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', 80)))
    count = int(collection.size().getInfo())
    if count == 0:
        raise RuntimeError(
            f'No Sentinel-2 scenes from {start.date()} through {target.date()}')

    composite = collection.map(mask_clouds).median().divide(10000).clip(aoi)
    metadata = {
        'start': start.strftime('%Y-%m-%d'),
        'end': target.strftime('%Y-%m-%d'),
        'count': count,
    }
    print(f"Sentinel-2: {metadata['start']} to {metadata['end']} "
          f"({count} source scenes)")
    return composite, aoi, metadata


def add_score_deciles(frame):
    """Assign distributional deciles without changing the model scores."""
    result = frame.copy()
    score = result['prob'].to_numpy(float)
    if not np.isfinite(score).all():
        raise ValueError('prob contains non-finite values')
    if np.any((score < 0) | (score > 1)):
        raise ValueError('prob must lie in [0, 1]')
    percentile_rank = pd.Series(score).rank(method='average', pct=True).to_numpy()
    result['score_decile'] = np.clip(
        np.ceil(percentile_rank * 10).astype(np.int16), 1, 10)
    result['score_decile_label'] = (
        'D' + result['score_decile'].astype(str) + ' (low to high score)')
    return result, np.quantile(score, np.linspace(0, 1, 11))


def canvas_dimensions(bounds, width_px):
    """Choose a locally distance-corrected aspect ratio for a static map."""
    west, south, east, north = bounds
    mean_lat = (south + north) / 2
    ratio = ((north - south) / max(east - west, 1e-12)
             / max(np.cos(np.radians(mean_lat)), 1e-6))
    return int(width_px), max(500, int(round(width_px * ratio)))


def prediction_overlay(frame, bounds, width, height, opacity=0.78):
    """Rasterize all footprint deciles on the exact GEE thumbnail canvas."""
    source = frame.to_crs(4326).reset_index(drop=True)
    values = source['score_decile'].to_numpy(np.uint8)
    areas = source.to_crs(source.estimate_utm_crs()).geometry.area.to_numpy()
    order = np.argsort(-areas)  # large polygons first; small ones remain visible
    shapes = [(source.geometry.iloc[i], int(values[i])) for i in order
              if source.geometry.iloc[i] is not None
              and not source.geometry.iloc[i].is_empty]
    transform = from_bounds(*bounds, width=width, height=height)
    deciles = rasterize(
        shapes, out_shape=(height, width), transform=transform,
        fill=0, all_touched=True, dtype='uint8')

    palette = np.asarray([
        tuple(int(color[j:j + 2], 16) for j in (1, 3, 5))
        for color in REFERENCE_PALETTE
    ], dtype=np.uint8)
    rgba = np.zeros((height, width, 4), dtype=np.uint8)
    occupied = deciles > 0
    rgba[occupied, :3] = palette[deciles[occupied] - 1]
    rgba[occupied, 3] = round(255 * float(opacity))
    return Image.fromarray(rgba, mode='RGBA')


def image_data_uri(image):
    buffer = io.BytesIO()
    image.save(buffer, format='PNG', optimize=True)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()


def add_ee_layer(map_widget, image, vis, name):
    map_id = ee.Image(image).getMapId(vis)
    folium.TileLayer(
        tiles=map_id['tile_fetcher'].url_format,
        attr='Sentinel-2 surface reflectance via Google Earth Engine',
        name=name, overlay=False, control=True, show=True,
    ).add_to(map_widget)


def decile_legend_html(quantiles):
    swatches = ''.join(
        f'<span title="D{i + 1}: {quantiles[i]:.3f}–{quantiles[i + 1]:.3f}" '
        f'style="display:inline-block;width:22px;height:12px;'
        f'background:{color}"></span>'
        for i, color in enumerate(REFERENCE_PALETTE))
    return f"""<div style="position:fixed;bottom:28px;left:28px;z-index:9999;
        background:rgba(255,255,255,.92);border:1px solid #666;
        padding:9px 11px;font:12px sans-serif;box-shadow:0 1px 5px #5558">
      <div style="font-weight:700;margin-bottom:5px">Prediction-score decile</div>
      <div style="line-height:0">{swatches}</div>
      <div style="display:flex;justify-content:space-between;width:220px;
        margin-top:4px"><span>low</span><span>high</span></div>
    </div>"""


def build_gee_map(frame, target_date, quantiles):
    """Interactive GEE overview styled after the reference figure."""
    bounds = padded_bounds(frame, MAP_PAD_FRACTION)
    width, height = canvas_dimensions(bounds, min(FIGURE_WIDTH_PX, 1800))
    optical, _, imagery = sentinel2_composite(
        bounds, target_date, GEE_LOOKBACK_DAYS)
    overlay = prediction_overlay(
        frame, bounds, width, height, OVERLAY_OPACITY)
    west, south, east, north = bounds
    map_widget = folium.Map(
        location=[(south + north) / 2, (west + east) / 2],
        zoom_start=12, tiles=None, control_scale=True, prefer_canvas=True)
    add_ee_layer(
        map_widget, optical,
        {'bands': ['B4', 'B3', 'B2'], 'min': 0.025,
         'max': 0.34, 'gamma': 1.18},
        f"Sentinel-2 {imagery['start']} to {imagery['end']}")
    folium.raster_layers.ImageOverlay(
        image=image_data_uri(overlay),
        bounds=[[south, west], [north, east]],
        name='prediction-score deciles', opacity=1, zindex=3,
        interactive=False, cross_origin=False).add_to(map_widget)
    map_widget.get_root().html.add_child(
        folium.Element(decile_legend_html(quantiles)))
    Fullscreen(position='topright').add_to(map_widget)
    folium.LayerControl(collapsed=False).add_to(map_widget)
    map_widget.fit_bounds([[south, west], [north, east]])

    output = prediction_path.with_name(prediction_path.stem + '_gee_map.html')
    map_widget.save(str(output))
    print(f'wrote interactive GEE map to {output}')
    return map_widget, optical, bounds, overlay


predictions, SCORE_QUANTILES = add_score_deciles(predictions)
GEE_MAP, GEE_OPTICAL, MAP_BOUNDS, PREDICTION_OVERLAY = build_gee_map(
    predictions, MAP_DATE, SCORE_QUANTILES)
GEE_MAP


### Publication-style PNG

The same Earth Engine composite and footprint overlay are rendered to a clean
static canvas with a panel label and metric scale bar. Set `PANEL_LABEL = None`
in the configuration cell when the output is not part of a multi-panel figure.


In [ ]:
def gee_safe_dimensions(width, height, max_uncompressed_mb=42):
    """Fit a three-band uint8 thumbnail below GEE request limits."""
    width, height = int(width), int(height)
    max_pixels = int(max_uncompressed_mb * 1024 ** 2 / 3)
    scale = min(
        1.0,
        np.sqrt(max_pixels / max(width * height, 1)),
        30000 / max(width, height),
    )
    return max(1, int(width * scale)), max(1, int(height * scale))


def earth_engine_thumbnail(image, bounds, width, height, attempts=3):
    """Download the GEE background, retrying rejected large requests."""
    rendered = image.visualize(
        bands=['B4', 'B3', 'B2'], min=0.025, max=0.34, gamma=1.18)
    west, south, east, north = bounds
    region = [
        [west, south], [east, south], [east, north],
        [west, north], [west, south],
    ]
    width, height = gee_safe_dimensions(width, height)

    for attempt in range(1, int(attempts) + 1):
        url = rendered.getThumbURL({
            'region': region,
            'dimensions': f'{width}x{height}',
            'crs': 'EPSG:4326',
            'format': 'png',
        })
        response = requests.get(url, timeout=180)
        if response.ok:
            background = Image.open(
                io.BytesIO(response.content)).convert('RGBA')
            if background.size != (width, height):
                background = background.resize(
                    (width, height), Image.Resampling.LANCZOS)
            return background, width, height

        detail = response.text[:1200].strip()
        if attempt == attempts:
            raise RuntimeError(
                f'GEE thumbnail failed after {attempts} attempts at '
                f'{width}x{height} pixels: HTTP {response.status_code}; '
                f'{detail}')
        width = max(256, int(width * 0.75))
        height = max(256, int(height * 0.75))
        print(f'GEE rejected the thumbnail; retrying at {width}x{height}.')



def choose_scale_bar(bounds):
    west, south, east, north = bounds
    width_km = (east - west) * 111.32 * np.cos(np.radians((south + north) / 2))
    for candidate in (20, 10, 5, 2, 1, 0.5):
        if candidate <= width_km / 4:
            return candidate, candidate / width_km
    return 0.2, 0.2 / width_km


def export_reference_figure(image, frame, target_date, path):
    """Export a clean, high-resolution figure matching the reference layout."""
    bounds = padded_bounds(frame, MAP_PAD_FRACTION)
    width, height = canvas_dimensions(bounds, FIGURE_WIDTH_PX)
    background, width, height = earth_engine_thumbnail(
        image, bounds, width, height)
    overlay = prediction_overlay(
        frame, bounds, width, height, OVERLAY_OPACITY)
    composed = Image.alpha_composite(background, overlay)

    dpi = 200
    fig, ax = plt.subplots(figsize=(width / dpi, height / dpi), dpi=dpi)
    ax.imshow(composed)
    ax.set_axis_off()
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#555555')
        spine.set_linewidth(0.6)

    scale_km, fraction = choose_scale_bar(bounds)
    x1, x2, y = 0.76, min(0.96, 0.76 + fraction), 0.045
    ax.plot([x1, x2], [y, y], transform=ax.transAxes,
            color='black', linewidth=3.0, solid_capstyle='butt')
    ax.plot([x1, x1], [y - .008, y + .008], transform=ax.transAxes,
            color='black', linewidth=1.2)
    ax.plot([x2, x2], [y - .008, y + .008], transform=ax.transAxes,
            color='black', linewidth=1.2)
    ax.text(x1, y + .012, '0', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8)
    ax.text(x2, y + .012, f'{scale_km:g} km', transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8)
    if PANEL_LABEL:
        ax.text(-.047, .985, PANEL_LABEL, transform=ax.transAxes,
                ha='left', va='top', fontsize=22, color='black')

    fig.savefig(path, bbox_inches='tight', pad_inches=0.18,
                facecolor='white')
    plt.show()
    print(f'wrote publication-style figure to {path}')
    return path


FIGURE_PATH = prediction_path.with_name(
    prediction_path.stem + '_gee_reference.png')
export_reference_figure(
    GEE_OPTICAL, predictions, MAP_DATE, FIGURE_PATH)


## 4. Lonboard error diagnostics

Hover a footprint for its score, score decile, observed/predicted condition,
split, date, and pipeline. The error encoding is deliberately diagnostic rather
than publication-oriented: **FP = cyan**, **FN = yellow**, while correct
predictions are muted green/red. Set `LONBOARD_ERRORS_ONLY = True` to isolate
mistakes.


In [ ]:
# Toggle this to inspect only mistakes; keep False for whole-city context.
LONBOARD_ERRORS_ONLY = False

diagnostic = predictions.to_crs(4326).copy()
diagnostic['observed'] = np.where(
    diagnostic['class'].to_numpy(int) == 1, 'damaged', 'intact')
diagnostic['predicted'] = np.where(
    diagnostic['pred'].to_numpy(int) == 1, 'damaged', 'intact')
diagnostic['outcome'] = np.select(
    [
        (diagnostic['class'] == 1) & (diagnostic['pred'] == 1),
        (diagnostic['class'] == 0) & (diagnostic['pred'] == 0),
        (diagnostic['class'] == 0) & (diagnostic['pred'] == 1),
        (diagnostic['class'] == 1) & (diagnostic['pred'] == 0),
    ],
    ['TP', 'TN', 'FP', 'FN'],
    default='NA',
)
diagnostic['score'] = diagnostic['prob'].round(4)
diagnostic['threshold'] = float(FINAL_THRESHOLD)

if LONBOARD_ERRORS_ONLY:
    diagnostic = diagnostic[diagnostic['outcome'].isin(['FP', 'FN'])].copy()
if diagnostic.empty:
    raise ValueError('The selected Lonboard filter produced no buildings')

# Muted correct predictions; bright cyan/yellow errors for rapid diagnosis.
OUTCOME_COLORS = {
    'TN': [0, 122, 77, 105],
    'TP': [217, 77, 97, 120],
    'FP': [0, 220, 255, 235],
    'FN': [255, 215, 0, 235],
}
fill_colors = np.asarray(
    [OUTCOME_COLORS[value] for value in diagnostic['outcome']],
    dtype=np.uint8)
line_colors = fill_colors.copy()
line_colors[:, 3] = 255

tooltip_columns = [name for name in [
    'geometry', 'outcome', 'score', 'score_decile', 'observed', 'predicted',
    'threshold', 'cnn_prob', 'split', 'city', 'date', 'pipeline',
] if name in diagnostic.columns]

diagnostic_layer = PolygonLayer.from_geopandas(
    diagnostic[tooltip_columns],
    get_fill_color=fill_colors,
    get_line_color=line_colors,
    line_width_min_pixels=0.6,
)
LONBOARD_MAP = LonboardMap(
    diagnostic_layer,
    show_tooltip=True,
    show_side_panel=True,
    picking_radius=6,
    height=720,
)
lonboard_path = prediction_path.with_name(
    prediction_path.stem + '_lonboard_diagnostics.html')
LONBOARD_MAP.to_html(lonboard_path, title=f'{CITY} prediction diagnostics')
print(f'wrote Lonboard diagnostics to {lonboard_path}')
LONBOARD_MAP


## Output contract

All outputs are written beneath the selected experiment's `predictions/`
folder: scored GeoJSON, interactive GEE HTML, publication PNG, and Lonboard
diagnostics HTML. Notebook 3 reads the frozen selected model and validation-
fitted threshold and does not modify training, validation, or test metrics.
